# Leksyon 02 - Pagsusuri sa Microsoft Agent Framework

Ang **Microsoft Agent Framework (MAF)** ay isang pinag-isang balangkas para sa paggawa ng mga AI agent. Nagbibigay ito ng malinis, nakokombinang arkitektura na may apat na pangunahing sangkap:

- **Kliyente** – kumokonekta sa isang AI model endpoint at humahawak ng komunikasyon
- **Ahente** – binabalutan ang isang kliyente ng mga tagubilin at mga depinisyon ng kasangkapan
- **Mga Kasangkapan** – pinalalawak ang kakayahan ng ahente gamit ang mga pasadyang function na maaaring tawagin ng modelo
- **Sesyon** – nagpapanatili ng kasaysayan ng pag-uusap para sa mga multi-turn na interaksyon

Sa leksyong ito, gagawa tayo ng isang **travel booking agent** na sumusuri ng kakayahan ng destinasyon gamit ang mga konseptong ito.


## Setup


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## Pag-unawa sa Arkitektura ng Agent Framework

Ang Microsoft Agent Framework ay sumusunod sa isang layered na arkitektura:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Client** – Isang `FoundryChatClient` na kumokonekta sa isang Azure OpenAI deployment. Ito ang humahawak ng authentication, pag-format ng kahilingan, at pag-parse ng tugon.
2. **Agent** – Ginawa mula sa client sa pamamagitan ng `provider.create_agent()`, pinagsasama ng agent ang access sa modelo kasama ang mga tagubilin (system prompt) at mga tool.
3. **Tools** – Mga Python function na may dekorasyong `@tool` na maaaring tawagin ng agent upang magsagawa ng mga aksyon o kumuha ng data.
4. **Session** – Isang `AgentSession` na object (ginawa sa pamamagitan ng `agent.create_session()`) na nag-iimbak ng kasaysayan ng pag-uusap, na nagpapahintulot ng multi-turn dialogue kung saan naaalala ng agent ang naunang konteksto.

Itayo natin ang bawat layer nang sunud-sunod.


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Pagdaragdag ng Mga Tool gamit ang @tool Decorator

Pinapayagan ng mga tool ang mga ahente na gumawa ng mga aksyon bukod sa pagbuo ng teksto. Ang `@tool` decorator ay nagko-convert ng isang regular na Python function sa isang bagay na maaaring tawagan ng ahente.

Mahahalagang puntos:
- Gumamit ng `Annotated[type, "description"]` upang maintindihan ng modelo ang bawat parameter.
- Ang docstring ang nagiging paglalarawan ng tool na nakikita ng modelo.
- Ang `approval_mode="never_require"` ay nangangahulugang ang tool ay tumatakbo nang awtomatiko nang walang kumpirmasyon ng user.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Gumagawa ng Ahente na may Mga Kasangkapan

Ngayon pinag-iisa natin ang kliyente, mga tagubilin, at mga kasangkapan sa isang ahente. Ang `instructions` ay nagsisilbing sistema ng prompt — tinutukoy nila ang persona at asal ng ahente.


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Maramihang Talakayan na may Mga Sesyon

Isang `AgentSession` (nilikha sa pamamagitan ng `agent.create_session()`) ang nagtatala ng lahat ng mga mensahe sa isang pag-uusap. Sa pamamagitan ng pagpasa ng parehong sesyon sa bawat tawag ng `agent.run()`, may akses ang agent sa buong kasaysayan ng pag-uusap at maaaring balikan ang mga naunang mensahe.

Pinapasa namin ang `tools=[check_destination_availability]` upang ang agent ay makatawag ng aming checker ng availability sa bawat turn.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Buod

Sa araling ito, iyong sinuri ang apat na haligi ng Microsoft Agent Framework:

| Konsepto | Ano ang Iyong Natutunan |
|---------|------------------|
| **Kliyente** | `FoundryChatClient` ay kumokonekta sa Azure OpenAI gamit ang credential-based na awtentikasyon |
| **Agent** | `provider.create_agent()` ay nagsasama ng koneksyon ng modelo kasama ang mga instruksiyon at pangalan |
| **Mga Kagamitan** | Ang `@tool` na dekorator ay nagpapakita ng mga Python na function para tawagin ng ahente |
| **Session** | `agent.create_session()` ay nagpapanatili ng kasaysayan ng pag-uusap sa maraming mga turno |

Ang mga batayang ito ay nagsasama-sama upang lumikha ng mga ahente na maaaring magkaroon ng natural na mga pag-uusap, tumawag ng panlabas na mga function, at mapanatili ang konteksto — ang pundasyon para sa mas advanced na mga pattern ng ahente sa mga susunod na aralin.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Pagtatanggi**:
Ang dokumentong ito ay isinalin gamit ang serbisyo ng AI translation na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagama't nagsusumikap kami para sa katumpakan, pakatandaan na ang awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o hindi pagkakatugma. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na pangunahing sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin ng tao. Hindi kami mananagot sa anumang maling pagkakaintindi o maling interpretasyon na nagmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
